# Kaggle SAKT Training Pipeline

> **IMPORTANT KAGGLE SETUP:**
> Configure your Kaggle Notebook settings before running:
> 1. Set Accelerator to **GPU T4x2**.
> 2. Add the Riiid dataset using `Add Data` -> Search "Riiid Answer Correctness Prediction"
>
> *(Note: We use Kaggle's standard input dataset paths in the code below. If you use a custom path locally, please update the input paths appropriately)*.

In [ ]:
import json
import os
from pathlib import Path
import pandas as pd
import torch
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split


def find_kaggle_train_csv():
    """Find train.csv in Kaggle input mounts."""
    candidates = [
        "/kaggle/input/riiid-answer-correctness-prediction/train.csv",
        "/kaggle/input/riiid-test-answer-prediction/train.csv",
    ]

    for c in candidates:
        if os.path.exists(c):
            return c

    input_root = Path("/kaggle/input")
    if input_root.exists():
        for p in input_root.glob("**/train.csv"):
            return str(p)

    raise FileNotFoundError(
        "Could not find train.csv under /kaggle/input. Add the Riiid dataset in Kaggle first."
    )


# BUMPED max_rows to 10 MILLION to prevent Transformer overfitting
def process_data(input_csv=None, output_dir="/kaggle/working/data/processed", max_seq_len=200, max_rows=10000000):
    if input_csv is None:
        input_csv = find_kaggle_train_csv()

    print(f"Loading data from {input_csv} (max {max_rows} rows)...")
    df = pd.read_csv(input_csv, nrows=max_rows)

    # Filter out lecture events; keep only question interactions.
    if "content_type_id" in df.columns:
        df = df[df.content_type_id == 0]

    df = df[["user_id", "content_id", "answered_correctly"]].copy()
    df.columns = ["user_id", "question_id", "answered_correctly"]

    print("Encoding question IDs...")
    q_unique = df["question_id"].unique()
    q_mapping = {q: i + 1 for i, q in enumerate(q_unique)}  # 0 reserved for padding
    df["question_id"] = df["question_id"].map(q_mapping)

    print("Grouping into sequences...")
    grouped = df.groupby("user_id").agg({
        "question_id": list,
        "answered_correctly": list,
    })

    sequences_q = [torch.tensor(q, dtype=torch.long) for q in grouped["question_id"]]
    sequences_a = [torch.tensor(a, dtype=torch.float32) for a in grouped["answered_correctly"]]

    print(f"Padding sequences to length {max_seq_len}...")
    sequences_q = [q[-max_seq_len:] for q in sequences_q]
    sequences_a = [a[-max_seq_len:] for a in sequences_a]

    padded_q = pad_sequence(sequences_q, batch_first=True, padding_value=0)
    padded_a = pad_sequence(sequences_a, batch_first=True, padding_value=-1)

    # Force exact width for consistent training tensor shapes.
    if padded_q.size(1) < max_seq_len:
        pad_size = max_seq_len - padded_q.size(1)
        padded_q = torch.cat([padded_q, torch.zeros(padded_q.size(0), pad_size, dtype=torch.long)], dim=1)
        padded_a = torch.cat([padded_a, torch.full((padded_a.size(0), pad_size), -1, dtype=torch.float32)], dim=1)

    print("Splitting dataset 80/10/10...")
    indices = list(range(padded_q.size(0)))
    train_idx, temp_idx = train_test_split(indices, test_size=0.2, random_state=42)
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42)

    metadata = {
        "input_csv": str(input_csv),
        "max_rows": max_rows,
        "max_seq_len": max_seq_len,
        "n_questions": len(q_unique),
        "question_id_mapping": {str(int(k)): int(v) for k, v in q_mapping.items()},
        "encoded_question_id_mapping": {str(int(v)): int(k) for k, v in q_mapping.items()},
    }

    print("Saving tensors and metadata...")
    os.makedirs(output_dir, exist_ok=True)
    torch.save({"q": padded_q[train_idx], "a": padded_a[train_idx]}, os.path.join(output_dir, "train.pt"))
    torch.save({"q": padded_q[val_idx], "a": padded_a[val_idx]}, os.path.join(output_dir, "val.pt"))
    torch.save({"q": padded_q[test_idx], "a": padded_a[test_idx]}, os.path.join(output_dir, "test.pt"))
    with open(os.path.join(output_dir, "metadata.json"), "w", encoding="utf-8") as f:
        json.dump(metadata, f)

    print(
        f"Preprocessing complete. Saved {len(train_idx)} train, {len(val_idx)} val, {len(test_idx)} test sequences."
    )
    print(f"Unique questions encoded: {len(q_unique)}")


# Run preprocessing in Kaggle
process_data()

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from torch.amp import autocast, GradScaler
from sklearn.metrics import roc_auc_score


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len, :]

class SAKT(nn.Module):
    # Maximum dropout (0.5) to brutally force generalization instead of memorization
    def __init__(self, n_questions, d_model=128, n_heads=8, dropout=0.5, max_seq_len=200):
        super().__init__()
        self.d_model = d_model
        self.n_questions = n_questions
        
        # Interaction embedding: Question ID + correctness (offset by n_questions)
        self.interaction_emb = nn.Embedding(2 * n_questions + 1, d_model, padding_idx=0)
        
        # Question embedding (for queries)
        self.question_emb = nn.Embedding(n_questions + 1, d_model, padding_idx=0)
        
        self.pos_encoder = PositionalEncoding(d_model, max_len=max_seq_len)
        
        self.attention = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads, dropout=dropout, batch_first=True)
        
        self.layer_norm1 = nn.LayerNorm(d_model)
        self.layer_norm2 = nn.LayerNorm(d_model)
        
        self.dropout = nn.Dropout(dropout)
        
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model)
        )
        
        self.pred_layer = nn.Linear(d_model, n_questions + 1)

    def forward(self, q, a, target_q=None):
        device = q.device
        
        interaction_tokens = q + self.n_questions * a.clamp(min=0).long()
        interaction_tokens = interaction_tokens.masked_fill(q == 0, 0)
        
        interactions = self.interaction_emb(interaction_tokens)
        interactions = self.pos_encoder(interactions)
        
        questions = self.question_emb(q)
        
        seq_len = q.size(1)
        attn_mask = torch.triu(torch.ones(seq_len, seq_len, device=device) * float('-inf'), diagonal=1)
        
        attn_out, _ = self.attention(
            query=questions, 
            key=interactions, 
            value=interactions, 
            attn_mask=attn_mask,
            need_weights=False
        )
        
        out = self.layer_norm1(questions + self.dropout(attn_out))
        ffn_out = self.ffn(out)
        out = self.layer_norm2(out + self.dropout(ffn_out))
        
        if target_q is not None:
            out = out[:, :target_q.size(1), :]
            target_q = target_q.long().clamp(min=0, max=self.n_questions)
            target_weight = self.pred_layer.weight[target_q]
            target_bias = self.pred_layer.bias[target_q]
            return (out * target_weight).sum(dim=-1) + target_bias

        logits = self.pred_layer(out) 
        return logits

    def predict_next(self, q_seq, a_seq, target_q):
        self.eval()
        with torch.no_grad():
            if not isinstance(q_seq, torch.Tensor):
                q_seq = torch.tensor(q_seq, dtype=torch.long).unsqueeze(0)
            if not isinstance(a_seq, torch.Tensor):
                a_seq = torch.tensor(a_seq, dtype=torch.float32).unsqueeze(0)
                
            logits = self(q_seq, a_seq)
            preds = torch.sigmoid(logits)
            last_step_preds = preds[0, -1, :]
            
            return last_step_preds[target_q].item()


def train_sakt():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Hyperparameters
    batch_size = 128  # Physical batch size; keep small enough for T4 attention backward.
    grad_accum_steps = 4  # Effective batch size is batch_size * grad_accum_steps.
    n_epochs = 20
    lr = 1e-3  # Dropped from 2e-3 to prevent the optimizer bouncing out of minima
    patience = 10 
    data_dir = "/kaggle/working/data/processed"

    print("Loading data...")
    train_data = torch.load(os.path.join(data_dir, "train.pt"), map_location="cpu", weights_only=True)
    val_data = torch.load(os.path.join(data_dir, "val.pt"), map_location="cpu", weights_only=True)

    # Determine num_questions dynamically from maximum ID in chunks
    n_questions = max(train_data['q'].max().item(), val_data['q'].max().item())
    print(f"Num questions encoded: {n_questions}")

    train_dataset = TensorDataset(train_data['q'], train_data['a'])
    val_dataset = TensorDataset(val_data['q'], val_data['a'])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=torch.cuda.is_available())
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, pin_memory=torch.cuda.is_available())

    model_config = {
        "n_questions": n_questions,
        "d_model": 128,
        "n_heads": 8,
        "dropout": 0.5,
        "max_seq_len": train_data['q'].size(1),
    }
    model = SAKT(**model_config)
    
    # Automatically load balance across both T4 GPUs dynamically on Kaggle
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs for parallel training!")
        model = nn.DataParallel(model)
        
    model = model.to(device)
    
    # Weight decay backed down slightly to 1e-2 to allow some parameter growth
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, 
        max_lr=lr, 
        steps_per_epoch=(len(train_loader) + grad_accum_steps - 1) // grad_accum_steps, 
        epochs=n_epochs,
        pct_start=0.3 # Shifted peak LR to Epoch 6 (~30%) to extend the warmup phase
    )
    
    criterion = nn.BCEWithLogitsLoss(reduction="none")
    scaler = GradScaler("cuda", enabled=torch.cuda.is_available())

    best_val_auc = 0.0
    epochs_no_improve = 0
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    for epoch in range(n_epochs):
        model.train()
        total_loss = 0.0
        optimizer.zero_grad(set_to_none=True)
        
        for step, (q_batch, a_batch) in enumerate(train_loader, start=1):
            q_batch = q_batch.to(device, non_blocking=True)
            a_batch = a_batch.to(device, non_blocking=True)
            
            with autocast("cuda", enabled=torch.cuda.is_available()):
                q_target = q_batch[:, 1:].long()
                a_target = a_batch[:, 1:]
                target_logits = model(q_batch, a_batch, target_q=q_target)
                
                mask = (a_target != -1) & (q_target > 0)
                
                valid_logits = target_logits[mask]
                valid_targets = a_target[mask]
                if valid_targets.numel() == 0:
                    continue
                
                # Heavier Label Smoothing (0.1) instead of (0.05) to penalize absolute certainty
                smoothed_targets = valid_targets * 0.90 + 0.05
                loss = criterion(valid_logits, smoothed_targets).mean()
                
            scaler.scale(loss / grad_accum_steps).backward()
            
            if step % grad_accum_steps == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
            
            total_loss += loss.item() * q_batch.size(0)
        
        train_loss = total_loss / len(train_dataset)

        model.eval()
        val_preds_all = []
        val_targets_all = []
        
        with torch.no_grad():
            for q_batch, a_batch in val_loader:
                q_batch = q_batch.to(device, non_blocking=True)
                a_batch = a_batch.to(device, non_blocking=True)
                
                q_target = q_batch[:, 1:].long()
                a_target = a_batch[:, 1:]
                mask = (a_target != -1) & (q_target > 0)
                
                target_logits = model(q_batch, a_batch, target_q=q_target)
                gathered_preds = torch.sigmoid(target_logits)
                
                valid_preds = gathered_preds[mask]
                valid_targets = a_target[mask]
                
                val_preds_all.extend(valid_preds.detach().cpu().numpy())
                val_targets_all.extend(valid_targets.detach().cpu().numpy())
                
        if len(set(val_targets_all)) < 2:
            print("Validation split has fewer than two target classes; skipping AUC for this epoch.")
            continue

        val_auc = roc_auc_score(val_targets_all, val_preds_all)
        print(f"Epoch {epoch+1:02d}/{n_epochs} | Loss: {train_loss:.4f} | Val AUC-ROC: {val_auc:.4f}")
        
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            # DataParallel wraps model layers in .module, so we unwrap it before saving
            state_dict = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
            torch.save(state_dict, "/kaggle/working/sakt_best.pt")
            torch.save({"state_dict": state_dict, "model_config": model_config, "best_val_auc": best_val_auc}, "/kaggle/working/sakt_best_checkpoint.pt")
            epochs_no_improve = 0
            print(f"  --> Saved new best model with Val AUC: {best_val_auc:.4f}")
        else:
            epochs_no_improve += 1
            print(f"  --> No improvement. Patience counter: {epochs_no_improve}/{patience}")
            if epochs_no_improve >= patience:
                print("Early stopping triggered. Training stopped.")
                break

train_sakt()

In [ ]:
import os
from IPython.display import FileLink, display

# When the SAKT model training fully reaches the epoch/patience limits
# this cell will allow you to quickly download the checkpoint and metadata.
for path in [
    "/kaggle/working/sakt_best.pt",
    "/kaggle/working/sakt_best_checkpoint.pt",
    "/kaggle/working/data/processed/metadata.json",
]:
    if os.path.exists(path):
        display(FileLink(path))